# 🌐 EidCloud AI Gateway Quickstart

This interactive notebook demonstrates the **EidCloud AI Gateway**, a model-agnostic unified AI reverse proxy built in pure PHP 8.2+.

In this quickstart, you will:
1. Setup and inspect the pure PHP gateway environment.
2. Configure model aliases (`coder`, `fast`, `general`) and execute unified OpenAI-compatible requests.
3. Simulate an automatic multi-provider failover when local Ollama is offline, falling back seamlessly to secondary models while recording token ledger accounting.

In [ ]:
# Cell 1: Environment Setup & Verifying CLI/Test Suite
!php -v
!php tests/run_tests.php

In [ ]:
# Cell 2: Programmatic Gateway Execution with Automatic Failover Demonstration
import subprocess
import json

php_script = '''<?php
require_once __DIR__ . '/src/Autoloader.php';

use EidCloud\\AiGateway\\Gateway;
use EidCloud\\AiGateway\\Providers\\MockProvider;

// 1. Initialize Gateway with a budget limit
$gateway = new Gateway(['budget_limit_usd' => 5.00]);

// 2. Setup mock providers to simulate local failure and cloud fallback
$primaryOllama = new MockProvider('ollama', true, '', 'Ollama local daemon connection refused (port 11434)');
$secondaryOpenAi = new MockProvider('openai', false, 'Hello from OpenAI fallback via EidCloud Gateway!');

$gateway->registerProvider($primaryOllama);
$gateway->registerProvider($secondaryOpenAi);

// 3. Define model alias mapping
$gateway->setAlias('coder', ['ollama/qwen2.5-coder', 'openai/gpt-4o-mini']);

// 4. Send chat completion to 'coder' alias
$requestPayload = [
    'model' => 'coder',
    'messages' => [
        ['role' => 'system', 'content' => 'You are an expert AI assistant.'],
        ['role' => 'user', 'content' => 'Write a function in PHP to reverse a string.']
    ],
    'temperature' => 0.7
];

echo "--- Initiating Unified Chat Request to alias 'coder' ---\n";
$response = $gateway->handleChatCompletion($requestPayload);

echo json_encode($response, JSON_PRETTY_PRINT | JSON_UNESCAPED_SLASHES) . "\n";

echo "\n--- Token Accounting Ledger Summary ---\n";
echo json_encode($gateway->getLedger()->getSummary(), JSON_PRETTY_PRINT | JSON_UNESCAPED_SLASHES) . "\n";
'''

with open('demo_failover.php', 'w') as f:
    f.write(php_script)

output = subprocess.check_output(['php', 'demo_failover.php'], text=True)
print(output)
